# E1–E2: LoRA Fine-Tuning and Rank Sweep — CheXpert

E1 = reproduce standard LoRA (r=48). E2 = rank sweep over [4, 8, 16, 32, 48] to evaluate adaptation capacity scaling.

In [ ]:
"""
Colab environment setup: clone repository, install dependencies, mount Google Drive.
"""
import os
import subprocess

if not os.path.exists("DinoV2B"):
    subprocess.run(["git", "clone", "https://github.com/YOUR_ORG/DinoV2B.git"], check=True)

os.chdir("DinoV2B")

subprocess.run(["pip", "install", "-e", ".", "-r", "requirements.txt", "-q"], check=True)

from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/dinov2_peft"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 1. Configuration

Load CheXpert benchmark dataset configuration and default LoRA adapter parameters.

In [ ]:
"""
Load dataset configuration and default LoRA fine-tuning hyperparameters.
"""
from typing import Any
from src.utils.config import load_config

cfg: dict[str, Any] = load_config("configs/chexpert.yaml")
lora_cfg: dict[str, Any] = load_config("configs/peft/lora.yaml")
print("CheXpert Configuration:", cfg["dataset"])
print("Default LoRA Configuration:", lora_cfg["lora"])

## 2. Build Datasets

Construct training and validation dataloaders with standard CheXpert normalization and augmentation transforms.

In [ ]:
"""
Construct PyTorch DataLoaders for CheXpert training and validation splits.
"""
from src.data import build_chexpert_dataloaders, build_transforms

train_tf = build_transforms(224, cfg["preprocessing"]["normalize_mean"], cfg["preprocessing"]["normalize_std"], augment=True)
val_tf = build_transforms(224, cfg["preprocessing"]["normalize_mean"], cfg["preprocessing"]["normalize_std"], augment=False)
train_loader, val_loader = build_chexpert_dataloaders(cfg, train_tf, val_tf)
print(f"Train batches: {len(train_loader)}, Validation batches: {len(val_loader)}")

## 3. LoRA Model Construction

Define a model builder that applies PEFT LoRA adapters to the DINOv2 ViT-B/14 backbone and pairs it with a multi-label classification head.

In [ ]:
"""
Factory function to construct LoRA-adapted DINOv2 model with multi-label classification head.
"""
from typing import Any
import torch
import torch.nn as nn
from src.models import apply_lora, count_trainable_parameters, load_backbone, MultiLabelHead
from src.utils.config import load_config

class LoRAModel(nn.Module):
    """
    End-to-end model combining LoRA-adapted DINOv2 backbone and multi-label classification head.
    """
    def __init__(self, backbone: nn.Module, head: nn.Module) -> None:
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        Forward pass extracting CLS token and returning multi-label logits.
        """
        outputs = self.backbone(pixel_values=pixel_values)
        cls_token = outputs.last_hidden_state[:, 0, :]
        return self.head(cls_token)

def build_lora_model(lora_overrides: dict[str, Any] | None = None) -> LoRAModel:
    """
    Builds a LoRA-wrapped DINOv2 model with a multi-label classification head.
    Returns the full model with PEFT adapter applied to the backbone.
    """
    lora_config: dict[str, Any] = load_config("configs/peft/lora.yaml", overrides=lora_overrides)
    backbone, _ = load_backbone()
    backbone = apply_lora(backbone, lora_config["lora"])
    head = MultiLabelHead(in_features=768, num_labels=14)
    return LoRAModel(backbone, head)

## 4. E1 — Standard LoRA (r=48)

Train and evaluate standard LoRA with rank 48 matching the base paper's PEFT configuration.

In [ ]:
"""
Train standard LoRA model (r=48), evaluate validation AUROC, and persist checkpoint.
"""
from typing import Any
import torch
from torch.optim import AdamW
from src.evaluation import Evaluator
from src.training.losses import MultiLabelBCELoss
from src.utils.logging_utils import ExperimentLogger

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_e1: LoRAModel = build_lora_model({"lora": {"r": 48, "lora_alpha": 16}})
model_e1.to(device)

trainable_e1, total_e1 = count_trainable_parameters(model_e1)
print(f"E1 Trainable parameters: {trainable_e1:,} / {total_e1:,} ({100 * trainable_e1 / total_e1:.2f}%)")

criterion = MultiLabelBCELoss()
optimizer = AdamW(
    [
        {"params": [p for n, p in model_e1.backbone.named_parameters() if p.requires_grad], "lr": 2e-4},
        {"params": model_e1.head.parameters(), "lr": 5e-3},
    ],
    weight_decay=1e-4,
)

epochs: int = 10
for epoch in range(epochs):
    model_e1.train()
    running_loss: float = 0.0
    for batch in train_loader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)
        optimizer.zero_grad()
        logits = model_e1(pixel_values)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch [{epoch + 1:02d}/{epochs:02d}] - Loss: {running_loss / len(train_loader):.4f}")

evaluator = Evaluator(task="multilabel", label_names=cfg["dataset"]["label_names"])
results_e1: dict[str, Any] = evaluator.evaluate(model_e1, val_loader, device=str(device))
print(f"E1 (r=48) Mean AUROC: {results_e1['mean_auroc']:.4f}")

logger = ExperimentLogger()
logger.log(
    run_id="lora_r48_chexpert",
    dataset="CheXpert",
    backbone="DINOv2 ViT-B/14",
    method="LoRA",
    rank=48,
    trainable_params=trainable_e1,
    val_auroc=results_e1["mean_auroc"],
)

## 5. E2 — Rank Sweep

Execute an empirical rank sweep across ranks [4, 8, 16, 32, 48] keeping alpha=16 and tracking trainable capacity versus AUROC.

In [ ]:
"""
Execute controlled LoRA rank sweep over [4, 8, 16, 32, 48] and record validation performance.
"""
from typing import Any
import pandas as pd
from torch.optim import AdamW

RANK_SWEEP: list[int] = [4, 8, 16, 32, 48]
sweep_results: list[dict[str, Any]] = []

for r in RANK_SWEEP:
    print(f"\n=== Running Rank Sweep: r={r} ===")
    model_r: LoRAModel = build_lora_model({"lora": {"r": r, "lora_alpha": 16}})
    model_r.to(device)
    trainable, total = count_trainable_parameters(model_r)

    opt = AdamW(
        [
            {"params": [p for n, p in model_r.backbone.named_parameters() if p.requires_grad], "lr": 2e-4},
            {"params": model_r.head.parameters(), "lr": 5e-3},
        ],
        weight_decay=1e-4,
    )

    for epoch in range(epochs):
        model_r.train()
        for batch in train_loader:
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)
            opt.zero_grad()
            loss = criterion(model_r(pixel_values), labels)
            loss.backward()
            opt.step()

    eval_out = evaluator.evaluate(model_r, val_loader, device=str(device))
    mean_auc: float = float(eval_out["mean_auroc"])
    print(f"Rank r={r} Completed -> Mean AUROC: {mean_auc:.4f}, Trainable: {trainable:,}")

    rec = {"rank": r, "trainable_params": trainable, "param_pct": 100 * trainable / total, "val_auroc": mean_auc}
    sweep_results.append(rec)
    logger.log(
        run_id=f"lora_r{r}_chexpert",
        dataset="CheXpert",
        backbone="DINOv2 ViT-B/14",
        method="LoRA",
        rank=r,
        trainable_params=trainable,
        val_auroc=mean_auc,
    )

## 6. Rank Sweep Results Table

Tabulate rank sweep outcomes and plot the validation AUROC curve against adaptation rank.

In [ ]:
"""
Tabulate rank sweep findings and render validation AUROC versus rank plot.
"""
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

results_df: pd.DataFrame = pd.DataFrame(sweep_results)
print(results_df.to_string(index=False))

sns.set_theme(style="whitegrid")
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.set_xlabel("LoRA Rank (r)", fontsize=12)
ax1.set_ylabel("Validation Mean AUROC", color="tab:blue", fontsize=12)
ax1.plot(results_df["rank"], results_df["val_auroc"], marker="o", linewidth=2.5, color="tab:blue", label="Val AUROC")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.set_ylabel("Trainable Parameters", color="tab:gray", fontsize=12)
ax2.bar(results_df["rank"], results_df["trainable_params"], alpha=0.25, width=2.0, color="tab:gray", label="Params")
ax2.tick_params(axis="y", labelcolor="tab:gray")
ax2.grid(False)

plt.title("E2: LoRA Rank Sweep Performance Trade-Off (CheXpert)", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## 7. Best Checkpoint Selection

Select the optimal LoRA rank configuration on validation data and designate the corresponding checkpoint target.

In [ ]:
"""
Identify optimal rank checkpoint by validation AUROC and designate checkpoint path.
"""
best_row = results_df.loc[results_df["val_auroc"].idxmax()]
best_rank: int = int(best_row["rank"])
print(f"Selected Best Configuration: Rank r={best_rank} with Validation AUROC: {best_row['val_auroc']:.4f}")

best_ckpt_path: str = f"{CHECKPOINT_DIR}/best_lora_r{best_rank}_chexpert.pt"
print(f"Target best checkpoint path: {best_ckpt_path}")